### **DATA Preprocessing** : Denoising and segmentation of T2 Transversal Lumbar MRI 

In [1]:
# Necessary imports

import os
import nibabel as nib
import numpy as np
from collections import Counter
import SimpleITK as sitk
from pathlib import Path
import re

#### Step 1 : Dataset structuration "Lumbar Spine MRI Dataset" from "https://data.mendeley.com/datasets/k57fr854j2/2"

The dataset is homogenous enough, geometrically in axial view, making the denoising slice-wise easier. 

Axial slices of the last disc sometimes follow the curvature of the column, while others are acquired in parallel blocks. making the transversal slices unequivalent geometrically.

**As a consequence** : The models do not only learn to supress noise, but are going to learn on anatomic and aquisition variations. But the **.nii** volumes are not isotropic 3D volumes but only slices clinically aquired to explore different levels.

**The dataset** : This is a rich and realistic clinical dataset of 515 patients, focused on symptomatic lumbar spine conditions (not necesserily stenose), with a significant number of slices and reasonable axial homogeneity. It is well-suited for supervised denoising on lumbar MRI, especially in slice-wise, but it requires significant preprocessing.



In [ ]:

def lire_nii(path):
    image = sitk.ReadImage(path)
    return sitk.GetArrayFromImage(image)

def slice_index(fname):
    m = re.search(r'_(\d+)\.ima$', fname.lower())
    return int(m.group(1)) if m else 10**9


# Extraction : creation of 3D MRIs from ima files
def extract(origin, destination, num_patient):

    # Transversal T2 - patient folder 
    patient_imas = sorted(
        [f for f in os.listdir(origin) if f.lower().endswith(".ima")],
        key=slice_index
    )

    slices_2d = []
    slice_shapes = []

    for ima in patient_imas:
        arr = lire_nii(os.path.join(origin, ima)) # (1,H,W) 
        
        # Riddance from 3rd unnecessary dimension
        if arr.ndim == 3 and arr.shape[0] == 1: 
            arr = arr[0]
        elif arr.ndim != 2:
            continue
        slice_shapes.append(arr.shape)
        slices_2d.append(arr)

    # common taille
    if len(slices_2d) < 2:
        return None

    # To harmonize the shape of all slices : we only keep the most common shape
    target_shape = Counter(slice_shapes).most_common(1)[0][0]

    before = len(slices_2d)
    slices_2d = [s for s, sh in zip(slices_2d, slice_shapes) if sh == target_shape]
    after = len(slices_2d)
    if after != before:
        print(f"[{num_patient}] removed {before-after} slices (shape mismatch), kept {after}")


    if len(slices_2d) < 2:
        return None

    vol = np.stack(slices_2d, axis=0)   # (Z,H,W)
    img3d = sitk.GetImageFromArray(vol)

    os.makedirs(destination, exist_ok=True)
    out = os.path.join(destination, num_patient + ".nii.gz")
    sitk.WriteImage(img3d, out)
    return out

# Base creation

def createBase_T2_TRA(base_path, destination):
    os.makedirs(destination, exist_ok=True)

    for patient_dossier in sorted(os.listdir(base_path)):
        patient_path = os.path.join(base_path, patient_dossier)
        if not os.path.isdir(patient_path):
            continue

        for sousDoss in os.listdir(patient_path):
            if "L-SPINE" not in sousDoss:
                continue

            sousPath = os.path.join(patient_path, sousDoss)

            for seq_name in os.listdir(sousPath):
                seq_path = os.path.join(sousPath, seq_name)

                if not os.path.isdir(seq_path):
                    continue

                # garder T2 transversal
                if seq_name.startswith("T2_TSE_TRA") and ("BOX" not in seq_name):
                    out_name = f"{patient_dossier}"  # unique, pas d'écrasement
                    extract(seq_path, destination, out_name)





In [ ]:
data = "data/01_MRI_Data/"
raw_dir = "raw_data"

os.makedirs(raw_dir, exist_ok=True)

createBase_T2_TRA(data, raw_dir)

In [9]:
import torch
from torch.utils.data import Dataset

In [ ]:
def add_rician_noise_raw(x, alpha=0.05, seed=None):
    """
    x: numpy array float32/float64 (raw intensities)
    alpha: intensité relative (0.02 easy, 0.05 medium, 0.08 hard)
    """
    if seed is not None:
        np.random.seed(seed)

    x = x.astype(np.float32)
    # Pencetiles
    p1, p99 = np.percentile(x, 1), np.percentile(x, 99)
    rng = max(p99 - p1, 1e-6)
    sigma = alpha * rng

    n1 = np.random.normal(0.0, sigma, size=x.shape).astype(np.float32)
    n2 = np.random.normal(0.0, sigma, size=x.shape).astype(np.float32)

    y = np.sqrt((x + n1)**2 + n2**2).astype(np.float32)
    return y



##### splitting for learning and inference

In [2]:
import random
import shutil

nii_dir = "raw_data"
out_dir = "raw_data_split"

train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1
seed = 42

files = [f for f in os.listdir(nii_dir) if f.endswith(".nii.gz")]
files.sort()

random.seed(seed)
random.shuffle(files)

n = len(files)
n_train = int(train_ratio * n)
n_val = int(val_ratio * n)

train_files = files[:n_train]
val_files = files[n_train:n_train + n_val]
test_files = files[n_train + n_val:]

train_dir = os.path.join(out_dir, "train")
val_dir = os.path.join(out_dir, "val")
test_dir = os.path.join(out_dir, "test")

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

for f in train_files:
    shutil.copy2(os.path.join(nii_dir, f), os.path.join(train_dir, f))

for f in val_files:
    shutil.copy2(os.path.join(nii_dir, f), os.path.join(val_dir, f))

for f in test_files:
    shutil.copy2(os.path.join(nii_dir, f), os.path.join(test_dir, f))


### Supervised denoising setting with synthetic noise : Rician noise
denoising in medical imaging aims to suppress/reduce/attenuate noise while preserving the important details and fine structures, texture and anatomic fidelity. 
The goal here is to find a compromise between noise suppression and preservation of important antomic structures.

In [ ]:
# The noise is synthetic and not real 
# From 3D MRIs to trainable 2D slices : noisy Vs clean (ground truth is not absolutely clean, it is already imperfect)
# An on-the-fly approach : each epoch sees a different noise, as slices are noised and used right away for training

class T2noise_dataset(Dataset):
    def __init__(self, clean_nii_dir, alpha_range=(0.02, 0.08)):
        self.paths = [
            os.path.join(clean_nii_dir, f)
            for f in os.listdir(clean_nii_dir)
            if f.endswith(".nii.gz")
        ] # All nii paths
        self.alpha_range = alpha_range # noise range

        # index: list (vol_id, slice_id) of all slices
        self.index = []
        for i, p in enumerate(self.paths):
            arr = sitk.GetArrayFromImage(sitk.ReadImage(p))  # (Z,H,W)
            z = arr.shape[0]
            for k in range(z):
                self.index.append((i, k))

    def __len__(self):
        return len(self.index) # number of samples 

    def __getitem__(self, idx):
        vol_i, k = self.index[idx]
        img = sitk.ReadImage(self.paths[vol_i])
        x = sitk.GetArrayFromImage(img)[k]  # clean

        alpha = np.random.uniform(*self.alpha_range)
        y = add_rician_noise_raw(x, alpha=alpha)

        # (1,H,W) tensors for pytorch
        x_t = torch.from_numpy(x).unsqueeze(0)
        y_t = torch.from_numpy(y).unsqueeze(0)

        return y_t, x_t